### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### CSV-Daten laden

Donald Trump Tweets bis 2021

In [2]:
tweets = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data/tweets.csv")

In [3]:
tweets.head()

,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged
0,98454970654916608,Republicans and Democrats have both created ou...,f,f,TweetDeck,49,255,2011-08-02 18:07:48,f
1,1234653427789070336,I was thrilled to be back in the Great city of...,f,f,Twitter for iPhone,73748,17404,2020-03-03 01:34:50,f
2,1218010753434820614,RT @CBS_Herridge: READ: Letter to surveillance...,t,f,Twitter for iPhone,0,7396,2020-01-17 03:22:47,f
3,1304875170860015617,The Unsolicited Mail In Ballot Scam is a major...,f,f,Twitter for iPhone,80527,23502,2020-09-12 20:10:58,f
4,1218159531554897920,RT @MZHemingway: Very friendly telling of even...,t,f,Twitter for iPhone,0,9081,2020-01-17 13:13:59,f


In [4]:
tweets_v1 = tweets.copy()

In [6]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
tweets_v1['timestamp'] = pd.to_datetime(tweets_v1['date'])

# 4. In UTC umrechnen
tweets_v1['timestamp_utc'] = tweets_v1['timestamp'].dt.tz_localize('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
tweets_v1['timestamp_utc_clean'] = tweets_v1['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [7]:
tweets_v1.head()

,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged,timestamp,timestamp_utc,timestamp_utc_clean
0,98454970654916608,Republicans and Democrats have both created ou...,f,f,TweetDeck,49,255,2011-08-02 18:07:48,f,2011-08-02 18:07:48,2011-08-02 18:07:48+00:00,2011-08-02 18:07:48
1,1234653427789070336,I was thrilled to be back in the Great city of...,f,f,Twitter for iPhone,73748,17404,2020-03-03 01:34:50,f,2020-03-03 01:34:50,2020-03-03 01:34:50+00:00,2020-03-03 01:34:50
2,1218010753434820614,RT @CBS_Herridge: READ: Letter to surveillance...,t,f,Twitter for iPhone,0,7396,2020-01-17 03:22:47,f,2020-01-17 03:22:47,2020-01-17 03:22:47+00:00,2020-01-17 03:22:47
3,1304875170860015617,The Unsolicited Mail In Ballot Scam is a major...,f,f,Twitter for iPhone,80527,23502,2020-09-12 20:10:58,f,2020-09-12 20:10:58,2020-09-12 20:10:58+00:00,2020-09-12 20:10:58
4,1218159531554897920,RT @MZHemingway: Very friendly telling of even...,t,f,Twitter for iPhone,0,9081,2020-01-17 13:13:59,f,2020-01-17 13:13:59,2020-01-17 13:13:59+00:00,2020-01-17 13:13:59


In [8]:
tweets_v1.dtypes

id                                   int64
text                                   str
isRetweet                              str
isDeleted                              str
device                                 str
favorites                            int64
retweets                             int64
date                                   str
isFlagged                              str
timestamp                   datetime64[us]
timestamp_utc          datetime64[us, UTC]
timestamp_utc_clean         datetime64[us]
dtype: object

WTI Future Kurse in USD von 2011-2022

In [9]:
wti = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data\wti.csv")

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_6352\1190269471.py:1: SyntaxWarning: invalid escape sequence '\w'
  wti = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data\wti.csv")


In [10]:
wti.head()

,time,open,high,low,close
0,2011-01-02 20:15:00,91.28,91.29,91.26,91.26
1,2011-01-02 20:16:00,91.26,91.26,91.25,91.26
2,2011-01-02 20:17:00,91.25,91.26,91.25,91.26
3,2011-01-02 20:18:00,91.27,91.27,91.26,91.26
4,2011-01-02 20:19:00,91.25,91.25,91.25,91.25


In [11]:
wti_v1 = wti.copy()

In [13]:
# Einbruch der counts um 17 Uhr deutet auf NY Zeitzone hin

# Mit diesem Python-Code gruppierst du die Daten nach der Uhrzeit, die aktuell im Datensatz steht, und zählst, wie viele Einträge es pro Stunde gibt:

# Fall 1: Die Lücke ist bei 17 Uhr. -> Deine Daten sind bereits in der New York Zeitzone (EST/EDT).

wti_v1['time'] = pd.to_datetime(wti_v1['time'])

# Stunde extrahieren (0 bis 23)
wti_v1['hour'] = wti_v1['time'].dt.hour

# Anzahl der Datenpunkte pro Stunde zählen
print(wti_v1['hour'].value_counts().sort_index())

hour
0     145365
1     154919
2     171426
3     182765
4     183397
5     182139
6     181676
7     182371
8     184237
9     185107
10    185036
11    185006
12    184443
13    180690
14    176731
15    167613
16    147228
17     19011
18    134967
19    129162
20    158876
21    161296
22    153402
23    146162
Name: count, dtype: int64


In [14]:
# Nur Freitage filtern - da Börsenschluss im 17 Uhr - deutet auf NY Zeit hin

# schauen wir uns den letzten Datenpunkt vor dem Wochenende an. Freitags um 17:00 Uhr New York

freitage = wti_v1[wti_v1['time'].dt.dayofweek == 4]

# Die jeweils letzte Uhrzeit an jedem Freitag anzeigen
print(freitage.groupby(freitage['time'].dt.date)['time'].max().head(10))

time
2011-01-07   2011-01-07 17:00:00
2011-01-14   2011-01-14 16:59:00
2011-01-21   2011-01-21 17:00:00
2011-01-28   2011-01-28 16:59:00
2011-02-04   2011-02-04 16:59:00
2011-02-11   2011-02-11 16:59:00
2011-02-18   2011-02-18 16:59:00
2011-02-25   2011-02-25 16:59:00
2011-03-04   2011-03-04 16:59:00
2011-03-11   2011-03-11 16:59:00
Name: time, dtype: datetime64[us]


In [15]:
wti_v2 = wti.copy()

In [16]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
wti_v2['time'] = pd.to_datetime(wti_v2['time'])

# 3. New York Zeit zuweisen (Lokalisieren)
# 'ambiguous='NaT'' fängt die doppelte Stunde bei der Umstellung im Herbst ab,
# 'nonexistent='NaT'' fängt die fehlende Stunde bei der Umstellung im Frühjahr ab.
wti_v2['timestamp_ny'] = wti_v2['time'].dt.tz_localize('America/New_York', ambiguous='NaT', nonexistent='NaT')

# 4. In UTC umrechnen
wti_v2['timestamp_utc'] = wti_v2['timestamp_ny'].dt.tz_convert('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
wti_v2['timestamp_utc_clean'] = wti_v2['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [18]:
wti_v2

,time,open,high,low,close,timestamp_ny,timestamp_utc,timestamp_utc_clean
0,2011-01-02 20:15:00,91.280,91.290,91.260,91.260,2011-01-02 20:15:00-05:00,2011-01-03 01:15:00+00:00,2011-01-03 01:15:00
1,2011-01-02 20:16:00,91.260,91.260,91.250,91.260,2011-01-02 20:16:00-05:00,2011-01-03 01:16:00+00:00,2011-01-03 01:16:00
2,2011-01-02 20:17:00,91.250,91.260,91.250,91.260,2011-01-02 20:17:00-05:00,2011-01-03 01:17:00+00:00,2011-01-03 01:17:00
3,2011-01-02 20:18:00,91.270,91.270,91.260,91.260,2011-01-02 20:18:00-05:00,2011-01-03 01:18:00+00:00,2011-01-03 01:18:00
4,2011-01-02 20:19:00,91.250,91.250,91.250,91.250,2011-01-02 20:19:00-05:00,2011-01-03 01:19:00+00:00,2011-01-03 01:19:00
...,...,...,...,...,...,...,...,...
3883020,2022-12-30 16:53:00,80.407,80.407,80.367,80.392,2022-12-30 16:53:00-05:00,2022-12-30 21:53:00+00:00,2022-12-30 21:53:00
3883021,2022-12-30 16:54:00,80.402,80.427,80.387,80.422,2022-12-30 16:54:00-05:00,2022-12-30 21:54:00+00:00,2022-12-30 21:54:00
3883022,2022-12-30 16:55:00,80.417,80.447,80.402,80.407,2022-12-30 16:55:00-05:00,2022-12-30 21:55:00+00:00,2022-12-30 21:55:00
3883023,2022-12-30 16:56:00,80.427,80.427,80.369,80.387,2022-12-30 16:56:00-05:00,2022-12-30 21:56:00+00:00,2022-12-30 21:56:00


In [22]:
wti_v2.dtypes

time                                     datetime64[us]
open                                            float64
high                                            float64
low                                             float64
close                                           float64
timestamp_ny           datetime64[us, America/New_York]
timestamp_utc                       datetime64[us, UTC]
timestamp_utc_clean                      datetime64[us]
dtype: object